# 👗 Kleding Classifier v2 — Nauwkeurig & Bulk

Dit notebook gebruikt **wargoninnovation/wargon-clothing-classifier**, een Vision Transformer (ViT) getraind op 30.000+ kledingfoto's met 27 specifieke categorieën.

De 27 klassen worden automatisch samengevoegd naar jouw 6 hoofdcategorieën:

| Jouw categorie | Wat het model herkent |
|---|---|
| 👕 T-shirt / Top | t-shirt, shirt, blouse, top, hoodie, sweater, tank top... |
| 👖 Broek / Jeans | jeans, trousers, shorts, leggings, joggers... |
| 🧥 Jas / Vest | jacket, coat, blazer, cardigan, vest, windbreaker... |
| 👗 Jurk / Rok | dress, skirt, mini dress, maxi dress... |
| 👟 Schoenen | shoes, boots, sneakers, sandals, heels... |
| 👜 Accessoires | bag, backpack, belt, scarf, hat, jewelry... |
| ❓ Overige | geen kleding herkend |

## Stap 1 — Installeren & importeren

Eenmalig uitvoeren om het model te downloaden (~350 MB).

In [22]:
# Installeer benodigde pakketten (verwijder # als je dit nog niet hebt)
# !pip install transformers torch pillow matplotlib

from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os
from pathlib import Path
from collections import defaultdict

print('✅ Imports geslaagd!')

✅ Imports geslaagd!


## Stap 2 — Model laden

Het model wordt de eerste keer gedownload (~350 MB). Daarna staat het lokaal op je pc gecached.

In [23]:
MODEL_NAAM = 'wargoninnovation/wargon-clothing-classifier'

print('⏳ Model laden... (eerste keer: ~350 MB download)')
processor = AutoImageProcessor.from_pretrained(MODEL_NAAM)
model     = AutoModelForImageClassification.from_pretrained(MODEL_NAAM)
model.eval()

# Bekijk alle beschikbare labels van het model
alle_labels = list(model.config.id2label.values())
print(f'\n✅ Model geladen! {len(alle_labels)} klassen beschikbaar:')
for i, label in enumerate(alle_labels):
    print(f'   {i:>2}. {label}')

⏳ Model laden... (eerste keer: ~350 MB download)


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]


✅ Model geladen! 27 klassen beschikbaar:
    0. LABEL_0
    1. LABEL_1
    2. LABEL_2
    3. LABEL_3
    4. LABEL_4
    5. LABEL_5
    6. LABEL_6
    7. LABEL_7
    8. LABEL_8
    9. LABEL_9
   10. LABEL_10
   11. LABEL_11
   12. LABEL_12
   13. LABEL_13
   14. LABEL_14
   15. LABEL_15
   16. LABEL_16
   17. LABEL_17
   18. LABEL_18
   19. LABEL_19
   20. LABEL_20
   21. LABEL_21
   22. LABEL_22
   23. LABEL_23
   24. LABEL_24
   25. LABEL_25
   26. LABEL_26


## Stap 3 — Categorieën koppelen

⚠️ **Belangrijk:** Run Stap 2 eerst. Bekijk welke labels het model heeft, en pas eventueel de mapping hieronder aan.

In [24]:
# Het model geeft LABEL_0 t/m LABEL_26 terug — hier koppelen we de echte namen eraan.
# Volgorde is alfabetisch zoals op de HuggingFace pagina van het model.
LABEL_NAMEN = {
    0:  'Blazer',
    1:  'Blouse',
    2:  'Cardigan',
    3:  'Dress',
    4:  'Hoodie',
    5:  'Jacket',
    6:  'Jeans',
    7:  'Nightgown',
    8:  'Outerwear',
    9:  'Pajamas',
    10: 'Rain jacket',
    11: 'Rain trousers',
    12: 'Robe',
    13: 'Shirt',
    14: 'Shorts',
    15: 'Skirt',
    16: 'Sweater',
    17: 'T-shirt',
    18: 'Tank top',
    19: 'Tights',
    20: 'Top',
    21: 'Training top',
    22: 'Trousers',
    23: 'Tunic',
    24: 'Vest',
    25: 'Winter jacket',
    26: 'Winter trousers',
}

LABEL_NAAR_CATEGORIE = {
    'Blazer':          'Jas / Vest',
    'Blouse':          'T-shirt / Top',
    'Cardigan':        'Jas / Vest',
    'Dress':           'Jurk / Rok',
    'Hoodie':          'T-shirt / Top',
    'Jacket':          'Jas / Vest',
    'Jeans':           'Broek / Jeans',
    'Nightgown':       'Jurk / Rok',
    'Outerwear':       'Jas / Vest',
    'Pajamas':         'T-shirt / Top',
    'Rain jacket':     'Jas / Vest',
    'Rain trousers':   'Broek / Jeans',
    'Robe':            'Jas / Vest',
    'Shirt':           'T-shirt / Top',
    'Shorts':          'Broek / Jeans',
    'Skirt':           'Jurk / Rok',
    'Sweater':         'T-shirt / Top',
    'T-shirt':         'T-shirt / Top',
    'Tank top':        'T-shirt / Top',
    'Tights':          'Broek / Jeans',
    'Top':             'T-shirt / Top',
    'Training top':    'T-shirt / Top',
    'Trousers':        'Broek / Jeans',
    'Tunic':           'T-shirt / Top',
    'Vest':            'Jas / Vest',
    'Winter jacket':   'Jas / Vest',
    'Winter trousers': 'Broek / Jeans',
}

KLEUREN = {
    'T-shirt / Top':  '#e8a87c',
    'Broek / Jeans':  '#6b8cba',
    'Jas / Vest':     '#7daa7d',
    'Jurk / Rok':     '#c47db5',
    'Schoenen':       '#e07070',
    'Accessoires':    '#d4a84b',
    'Overige':        '#aaaaaa',
    'Twijfel':        '#ff6b6b',
}

print('Categorieen geconfigureerd!')
print(f'   {len(LABEL_NAMEN)} model-labels gekoppeld aan categorieen + Twijfel')


✅ Categorieën geconfigureerd!
   54 model-labels gekoppeld aan 6 categorieën


## Stap 4 — Classificatiefunctie

In [25]:
# Drempel: alles onder dit percentage gaat naar "Twijfel" voor handmatige controle
DREMPEL = 0.35

def classificeer(img: Image.Image):
    """
    Classificeert een kledingafbeelding.
    Alles onder de drempel wordt als "Twijfel" gemarkeerd.
    """
    inputs = processor(images=img.convert('RGB'), return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=-1)[0]

    top5_kansen, top5_indices = torch.topk(probs, min(5, len(probs)))
    top5 = []
    for k, idx in zip(top5_kansen, top5_indices):
        model_label = LABEL_NAMEN.get(int(idx), f'LABEL_{int(idx)}')
        categorie   = LABEL_NAAR_CATEGORIE.get(model_label, 'Overige')
        top5.append({
            'model_label': model_label,
            'categorie':   categorie,
            'kans':        float(k),
        })

    beste = top5[0]

    # Onder de drempel = twijfelgeval, zelf nakijken
    if beste['kans'] < DREMPEL:
        categorie = 'Twijfel'
    else:
        categorie = beste['categorie']

    return {
        'label':       categorie,
        'model_label': beste['model_label'],
        'kans':        beste['kans'],
        'top5':        top5,
    }

print(f'Classificatiefunctie klaar! Drempel: {DREMPEL*100:.0f}%')
print(f'   Alles onder {DREMPEL*100:.0f}% zekerheid wordt als "Twijfel" gemarkeerd.')


✅ Classificatiefunctie klaar!


## Stap 5 — Bulk classificatie

Zet je foto's in een map genaamd **`kleding`** naast dit notebook.

In [26]:
# 🔧 Pas aan indien nodig
MAP = 'kleding'
EXTENSIES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

if not os.path.isdir(MAP):
    print(f'⚠️  Map "{MAP}" niet gevonden!')
    print(f'   Maak een map "{MAP}" aan naast dit notebook en zet daar je foto\'s in.')
else:
    bestanden = sorted([
        f for f in Path(MAP).iterdir()
        if f.suffix.lower() in EXTENSIES
    ])

    if not bestanden:
        print(f'⚠️  Geen afbeeldingen gevonden in "{MAP}"')
    else:
        print(f'📁 {len(bestanden)} afbeeldingen gevonden\n')
        resultaten = []

        for i, pad in enumerate(bestanden, 1):
            try:
                img = Image.open(pad)
                r = classificeer(img)
                resultaten.append({'pad': pad, 'img': img.copy(), **r})
                print(f'  [{i:>3}/{len(bestanden)}] {pad.name:35s} → {r["label"]:22s} '
                      f'(model: {r["model_label"]}, {r["kans"]*100:.0f}%)')
            except Exception as e:
                print(f'  ❌ {pad.name}: {e}')

        print(f'\n✅ Klaar! {len(resultaten)} afbeeldingen geclassificeerd.')

📁 16 afbeeldingen gevonden

  [  1/16] 20260114_110354-removebg-preview.png → ❓ Overige              (model: LABEL_5, 42%)
  [  2/16] 20260114_110506-removebg-preview.png → ❓ Overige              (model: LABEL_8, 18%)
  [  3/16] 20260114_110537-removebg-preview.png → ❓ Overige              (model: LABEL_0, 28%)
  [  4/16] 20260114_110604-removebg-preview.png → ❓ Overige              (model: LABEL_5, 26%)
  [  5/16] 20260114_110724-removebg-preview.png → ❓ Overige              (model: LABEL_1, 19%)
  [  6/16] 20260114_110727-removebg-preview.png → ❓ Overige              (model: LABEL_20, 30%)
  [  7/16] 20260114_110844-removebg-preview.png → ❓ Overige              (model: LABEL_0, 26%)
  [  8/16] 20260114_110934-removebg-preview.png → ❓ Overige              (model: LABEL_16, 43%)
  [  9/16] 20260114_111015-removebg-preview.png → ❓ Overige              (model: LABEL_16, 60%)
  [ 10/16] 20260114_111116-removebg-preview.png → ❓ Overige              (model: LABEL_8, 14%)
  [ 11/16] 20260114

## Stap 6 — Resultaten per categorie bekijken

In [20]:
if not resultaten:
    print('Geen resultaten - run Stap 5 eerst.')
else:
    groepen = defaultdict(list)
    for r in resultaten:
        groepen[r['label']].append(r)

    print(f'Samenvatting ({len(resultaten)} fotos):')
    volgorde = ['T-shirt / Top', 'Broek / Jeans', 'Jas / Vest', 'Jurk / Rok',
                'Schoenen', 'Accessoires', 'Overige', 'Twijfel']
    for label in volgorde:
        if label in groepen:
            twijfel_marker = ' <- nakijken!' if label == 'Twijfel' else ''
            print(f'   {label}: {len(groepen[label])}x{twijfel_marker}')

    print('\n' + '='*60)

    for label in volgorde:
        if label not in groepen:
            continue
        items = groepen[label]
        kleur = KLEUREN.get(label, '#aaaaaa')
        print(f'\n{label} - {len(items)} foto(s)')

        cols = min(4, len(items))
        rows = (len(items) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))

        if len(items) == 1:
            axes = [axes]
        elif rows == 1:
            axes = list(axes)
        else:
            axes = axes.flatten().tolist()

        for ax, item in zip(axes, items):
            ax.imshow(item['img'])
            ax.axis('off')
            ax.set_title(
                f"{item['pad'].name}\nModel: {item['model_label']}\n{item['kans']*100:.0f}% zekerheid",
                fontsize=8, color=kleur, fontweight='bold'
            )
            for spine in ax.spines.values():
                spine.set_edgecolor(kleur)
                spine.set_linewidth(3)
                spine.set_visible(True)

        for ax in axes[len(items):]:
            ax.set_visible(False)

        fig.suptitle(label, fontsize=14, fontweight='bold', color=kleur, y=1.01)
        plt.tight_layout()
        plt.show()


📊 Samenvatting (16 foto's):



## Stap 7 — Fout geclassificeerd? Handmatig corrigeren

Als het model iets fout heeft ingedeeld, kun je hier handmatig de categorie aanpassen.

In [21]:
# Handmatig een foto herindelen
# Verander BESTANDSNAAM en NIEUWE_CATEGORIE en run de cel

BESTANDSNAAM     = 'mijn_foto.jpg'
NIEUWE_CATEGORIE = 'T-shirt / Top'

# Beschikbare categorieen:
# 'T-shirt / Top'   'Broek / Jeans'   'Jas / Vest'
# 'Jurk / Rok'      'Schoenen'        'Accessoires'

for r in resultaten:
    if r['pad'].name == BESTANDSNAAM:
        oud = r['label']
        r['label'] = NIEUWE_CATEGORIE
        print(f'Gecorrigeerd: {BESTANDSNAAM}: {oud} -> {NIEUWE_CATEGORIE}')
        break
else:
    print(f'Bestand "{BESTANDSNAAM}" niet gevonden in resultaten.')

# Run Stap 6 opnieuw om het bijgewerkte overzicht te zien


⚠️  "mijn_foto.jpg" niet gevonden in resultaten.


---
## 💡 Tips

**Model herkent een label dat niet in LABEL_MAPPING staat?**  
Voeg het toe aan de mapping in Stap 3:
```python
'denim jacket': '🧥 Jas / Vest',
```

**Alles komt als 'Overige'?**  
Controleer in Stap 2 welke exacte labels het model gebruikt en pas de mapping in Stap 3 aan.

**Foto's met een persoon erin werken beter** dan losstaande kledingstukken op een witte achtergrond — het model is getraind op gedragen kleding.

**Volgende stap:** resultaten exporteren naar JSON voor gebruik in KledingHelper.